# Week 1 — Exploratory Data Analysis (EDA)

<a href="https://colab.research.google.com/github/Sushumna09/loan-approval-fairness/blob/main/notebooks/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Project**: Loan Approval Prediction with Fairness & Bias Analysis

**Dataset**: Home Credit Default Risk (307,511 loan applications, 122 features)

## Goal for Week 1

Before we build any model, we need to *understand* the data.

We'll answer:
1. How big is the dataset? What columns exist?
2. How many missing values? Which columns are most affected?
3. What percentage of applicants defaulted? (This tells us about **class imbalance**.)
4. What do the demographic distributions look like? (gender, age, income)
5. Are there obvious patterns between features and default risk?

**Rule for Week 1**: Don't try to build a model yet. Just observe, plot, and write insights.


## Setup

Import libraries. In Colab, these are all pre-installed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

print(f"pandas: {pd.__version__}")
print(f"numpy:  {np.__version__}")
print("Libraries ready.")

## Load the Dataset

Get `application_train.csv` from Kaggle: https://www.kaggle.com/c/home-credit-default-risk/data

**Two ways to make the file available to Colab** — pick one:

### Option A: Direct upload (simplest, must repeat each session)

1. Click the **folder icon** on the left sidebar of Colab
2. Click the **upload icon** (📤)
3. Select your `application_train.csv` file
4. Wait for upload to finish (~1-2 min for 166 MB)
5. Run the "Option A" cell below

### Option B: Google Drive (one-time setup, permanent across sessions)

1. Upload `application_train.csv` to your Google Drive, e.g. in a folder `MyDrive/data/`
2. Run the "Option B" cell below — it will ask you to authenticate Drive

Pick whichever you prefer.

In [ ]:
# ============================================================
# OPTION A — file uploaded directly to Colab
# ============================================================
# Uncomment and run these two lines if you used Option A above:

df = pd.read_csv('application_train.csv')

# ============================================================
# OPTION B — file in Google Drive at MyDrive/data/application_train.csv
# ============================================================
# Uncomment and run these three lines if you used Option B above:

# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/data/application_train.csv')

print(f"Dataset loaded! Shape: {df.shape}")

## 1. Basic Exploration

Let's get a feel for the dataset — size, columns, data types.

In [ ]:
print(f"Rows (applications): {df.shape[0]:,}")
print(f"Columns (features):  {df.shape[1]}")
print(f"\nColumn data types breakdown:")
print(df.dtypes.value_counts())

In [ ]:
# First few rows — always look at your data
df.head()

In [ ]:
# Statistical summary of numerical columns
df.describe()

## 2. Missing Values

Real-world data is messy. Let's see how much is missing.

In [ ]:
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_percent': missing_percent
}).sort_values('missing_percent', ascending=False)

missing_df = missing_df[missing_df['missing_count'] > 0]

print(f"Total columns:                    {df.shape[1]}")
print(f"Columns with any missing values:  {len(missing_df)}")
print(f"Columns 100% missing:             {(missing_df['missing_percent'] == 100).sum()}")
print(f"\nTop 20 columns with most missing values:")
missing_df.head(20)

In [ ]:
# Visualize the top missing columns
top_missing = missing_df.head(20)

plt.figure(figsize=(12, 6))
plt.barh(top_missing.index, top_missing['missing_percent'], color='steelblue')
plt.xlabel('Missing Percentage (%)')
plt.title('Top 20 Columns by Missing Value Percentage')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**Insight**: Many columns are heavily missing. Some are >60% empty. In Week 2, we'll need to decide: drop them, impute them, or let XGBoost handle them natively (XGBoost can handle missing values as-is).

## 3. Target Variable — The Big Question

- `TARGET = 1` → applicant **defaulted** (had payment difficulties)
- `TARGET = 0` → applicant **repaid** the loan successfully

This is what our model will predict.

In [ ]:
target_counts = df['TARGET'].value_counts()
default_rate = df['TARGET'].mean() * 100

print(f"Repaid (TARGET=0):    {target_counts[0]:,}  ({100-default_rate:.2f}%)")
print(f"Defaulted (TARGET=1): {target_counts[1]:,}  ({default_rate:.2f}%)")
print(f"\nImbalance ratio: 1 defaulter for every {target_counts[0] // target_counts[1]} repayers")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].bar(['Repaid (0)', 'Defaulted (1)'], target_counts.values,
          color=['seagreen', 'crimson'])
ax[0].set_title('Class Distribution (linear scale)')
ax[0].set_ylabel('Count')

ax[1].bar(['Repaid (0)', 'Defaulted (1)'], target_counts.values,
          color=['seagreen', 'crimson'])
ax[1].set_yscale('log')
ax[1].set_title('Class Distribution (log scale — reveals imbalance)')
ax[1].set_ylabel('Count (log)')

plt.tight_layout()
plt.show()

**Key insight**: About **92%** of applicants repaid, only **8%** defaulted. This is a severely imbalanced dataset.

**Implications**:
- **Accuracy is misleading**: A dumb model always predicting "will repay" gets ~92% accuracy but catches zero defaulters!
- We must use metrics like **Precision, Recall, F1, ROC-AUC, and especially PR-AUC** (which is designed for imbalanced problems).
- We'll need to handle imbalance during training (class weights, threshold tuning, etc.).

## 4. Gender Distribution

Gender is a **protected attribute** — we'll use it for fairness analysis in Week 3.

In [ ]:
gender_counts = df['CODE_GENDER'].value_counts()
print("Gender distribution:")
print(gender_counts)
print(f"\nNote: 'XNA' means unknown/missing gender. We'll drop these ({gender_counts.get('XNA', 0)} rows) in modeling.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

gender_counts.plot(kind='bar', ax=ax[0], color=['steelblue', 'coral', 'gray'])
ax[0].set_title('Overall Gender Distribution')
ax[0].set_ylabel('Count')
ax[0].tick_params(axis='x', rotation=0)

default_by_gender = df.groupby('CODE_GENDER')['TARGET'].mean() * 100
default_by_gender.plot(kind='bar', ax=ax[1], color=['steelblue', 'coral', 'gray'])
ax[1].set_title('Default Rate by Gender (%)')
ax[1].set_ylabel('Default Rate (%)')
ax[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(f"\nDefault rate by gender:")
print(default_by_gender.round(2))

## 5. Age Analysis

The `DAYS_BIRTH` column stores age as *negative days since birth*. Let's convert to years.

In [ ]:
df['AGE'] = (-df['DAYS_BIRTH'] // 365).astype(int)

print("Age statistics:")
print(df['AGE'].describe().round(1))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))

df['AGE'].hist(bins=30, ax=ax[0], color='steelblue', edgecolor='black')
ax[0].set_title('Age Distribution of Applicants')
ax[0].set_xlabel('Age')
ax[0].set_ylabel('Count')

df['AGE_GROUP'] = pd.cut(df['AGE'], bins=[20, 30, 40, 50, 60, 70],
                         labels=['20-30', '30-40', '40-50', '50-60', '60-70'])
default_by_age = df.groupby('AGE_GROUP')['TARGET'].mean() * 100
default_by_age.plot(kind='bar', ax=ax[1], color='coral')
ax[1].set_title('Default Rate by Age Group (%)')
ax[1].set_ylabel('Default Rate (%)')
ax[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(f"\nDefault rate by age group:")
print(default_by_age.round(2))

## 6. Income and Loan Amount

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Income distribution — heavily skewed, so use log scale
df['AMT_INCOME_TOTAL'].hist(bins=100, ax=axes[0,0], color='steelblue')
axes[0,0].set_yscale('log')
axes[0,0].set_xscale('log')
axes[0,0].set_title('Income Distribution (log-log)')
axes[0,0].set_xlabel('Total Income')

# Loan amount distribution
df['AMT_CREDIT'].hist(bins=50, ax=axes[0,1], color='coral')
axes[0,1].set_title('Loan Amount Distribution')
axes[0,1].set_xlabel('Loan Amount')

# Income by default status
df.boxplot(column='AMT_INCOME_TOTAL', by='TARGET', ax=axes[1,0])
axes[1,0].set_yscale('log')
axes[1,0].set_title('Income by Default Status (log scale)')
axes[1,0].set_xlabel('Default (1=yes, 0=no)')

# Loan amount by default status
df.boxplot(column='AMT_CREDIT', by='TARGET', ax=axes[1,1])
axes[1,1].set_title('Loan Amount by Default Status')
axes[1,1].set_xlabel('Default (1=yes, 0=no)')

plt.suptitle('')
plt.tight_layout()
plt.show()

print("\nIncome — median values:")
print(df.groupby('TARGET')['AMT_INCOME_TOTAL'].median())
print("\nLoan amount — median values:")
print(df.groupby('TARGET')['AMT_CREDIT'].median())

## 7. Education and Family Status

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

edu_default = df.groupby('NAME_EDUCATION_TYPE')['TARGET'].agg(['count', 'mean'])
edu_default['default_pct'] = edu_default['mean'] * 100
edu_default = edu_default.sort_values('default_pct')

ax[0].barh(edu_default.index, edu_default['default_pct'], color='steelblue')
ax[0].set_title('Default Rate by Education Level')
ax[0].set_xlabel('Default Rate (%)')

fam_default = df.groupby('NAME_FAMILY_STATUS')['TARGET'].agg(['count', 'mean'])
fam_default['default_pct'] = fam_default['mean'] * 100
fam_default = fam_default.sort_values('default_pct')

ax[1].barh(fam_default.index, fam_default['default_pct'], color='coral')
ax[1].set_title('Default Rate by Family Status')
ax[1].set_xlabel('Default Rate (%)')

plt.tight_layout()
plt.show()

print("Default rate by education:")
print(edu_default[['count', 'default_pct']].round(2))

## 8. External Credit Scores (EXT_SOURCE_*)

Three anonymized credit scores from external agencies. These are usually the **strongest** predictors.

In [ ]:
ext_sources = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

print("Correlation with TARGET (default):")
for src in ext_sources:
    corr = df[[src, 'TARGET']].corr().iloc[0, 1]
    print(f"  {src}: {corr:+.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, src in enumerate(ext_sources):
    df.boxplot(column=src, by='TARGET', ax=axes[i])
    axes[i].set_title(f'{src} by Default Status')
    axes[i].set_xlabel('Default (1=yes, 0=no)')

plt.suptitle('')
plt.tight_layout()
plt.show()

**Key insight**: All three external scores have negative correlation with default — meaning *higher external score = lower default risk*. This makes sense; these ARE credit scores. These will likely be the most important features in our model.

## 9. Summary of Findings

_(Fill in your observations after running all the cells above.)_

- **Dataset shape**: ___ rows × ___ columns
- **Missing values**: Many columns have significant gaps. Top offenders include ___
- **Class imbalance**: ~8% default rate — severely imbalanced
- **Gender**: More female applicants than male; default rates by gender: ___
- **Age**: Younger applicants (20-30) tend to have higher default rates
- **Income vs default**: Defaulters have slightly ___ median income
- **Education**: Higher education levels correlate with ___ default rates
- **External scores**: Strongest predictors — negative correlation with default

## Implications for Modeling

1. **Metrics** — accuracy is useless; use PR-AUC, ROC-AUC, F1, Recall
2. **Missing values** — many columns; XGBoost handles this natively (Week 2)
3. **Class imbalance** — use class weights or threshold tuning (Week 2)
4. **Fairness** — gender and age will be our protected attributes for Week 3 audit
5. **Feature importance** — EXT_SOURCE_* likely dominates; that's fine, but we'll check other features too

## Next Notebook

**`02_baseline_model.ipynb`** — Preprocess features and train a Logistic Regression baseline.